# ECON 4370 — Homework 3: A Simpler AI Economic Data Agent

**Due:** (instructor fills)  
**Submission:** Upload this `.ipynb` to Blackboard.

## Goal
Build a **simple AI agent** that can:
1. read a user question,
2. choose from a small set of approved economic series,
3. create a small dataset,
4. make a plot or run one simple regression,
5. return a short explanation in plain English.

This version is **more guided** than the in-class version. You will complete a structured template rather than build everything from scratch.


## 0) Setup

In [ ]:
# If needed, uncomment this:
# !pip -q install pandas numpy plotly statsmodels fredapi openai

import os
import json
from getpass import getpass

import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.formula.api as smf


## 1) Enter your API keys

In [ ]:
# OpenAI API key (required)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

print("OpenAI key loaded:", "YES" if os.environ.get("OPENAI_API_KEY") else "NO")


In [ ]:
# FRED API key (optional)
if not os.environ.get("FRED_API_KEY"):
    fred_key = getpass("Optional: paste your FRED API key, or press Enter to skip: ")
    if fred_key.strip():
        os.environ["FRED_API_KEY"] = fred_key.strip()

print("FRED key loaded:", "YES" if os.environ.get("FRED_API_KEY") else "NO")


## 2) Approved concepts and series

To keep this homework manageable, your agent will only work with these concepts.
You do **not** need to search all of FRED.


In [ ]:
SERIES = {
    "inflation": "CPIAUCSL",
    "core inflation": "CPILFESL",
    "unemployment": "UNRATE",
    "fed funds rate": "FEDFUNDS",
    "oil price": "DCOILWTICO",
    "wages": "CES0500000003"
}

SERIES_DESCRIPTIONS = {
    "CPIAUCSL": "Consumer Price Index (monthly)",
    "CPILFESL": "Core CPI (monthly)",
    "UNRATE": "Unemployment Rate (monthly)",
    "FEDFUNDS": "Federal Funds Rate (monthly)",
    "DCOILWTICO": "WTI Oil Price (daily)",
    "CES0500000003": "Average Hourly Earnings (monthly)"
}

SERIES


## 3) FRED helper functions

These functions are already written for you.
Your main job is to understand how they are used.


In [ ]:
from fredapi import Fred

def fred_series(series_id, start="1990-01-01", end=None):
    fred = Fred(api_key=os.environ.get("FRED_API_KEY"))
    s = fred.get_series(series_id, observation_start=start, observation_end=end)
    s.index = pd.to_datetime(s.index)
    s.name = series_id
    return s

def to_monthly(series):
    """If a series is daily, convert it to monthly average.
    If it is already monthly, keep it at monthly frequency.
    """
    s = series.dropna().copy()
    if len(s) >= 3:
        median_gap = np.median(np.diff(s.index.values).astype("timedelta64[D]").astype(int))
    else:
        median_gap = 31

    if median_gap < 20:
        s = s.resample("M").mean()

    s = s.resample("M").last()
    s.index = s.index.to_period("M").to_timestamp("M")
    return s

def yoy_pct(series):
    return 100 * (series / series.shift(12) - 1)


## 4) Small tool functions

These are the three tools your agent can use.

### Important
This homework is **not** about writing a huge agent framework.  
It is about using a few functions well.


In [ ]:
def build_dataset(series_list, start="1990-01-01"):
    frames = []
    for sid in series_list:
        s = fred_series(sid, start=start)
        s = to_monthly(s)
        frames.append(s)

    df = pd.concat(frames, axis=1)
    df.index.name = "date"
    return df

def make_plot(df, columns, title):
    fig = px.line(df, x=df.index, y=columns, title=title)
    fig.show()

def run_regression(df, formula):
    model = smf.ols(formula=formula, data=df).fit()
    print(model.summary())
    return {
        "formula": formula,
        "n_obs": int(model.nobs),
        "r2": float(model.rsquared),
        "params": {k: float(v) for k, v in model.params.items()}
    }


## 5) Very simple planner

Instead of asking the model to invent a full JSON workflow, we will make this easier.

The model only needs to return a small JSON object with:
- which concepts to use
- whether the task is `"plot"` or `"regression"`
- a short explanation

This keeps the coding level reasonable.


In [ ]:
from openai import OpenAI
client = OpenAI()

PLANNER_PROMPT = f"""
You are helping with a student economics homework assignment.

Use only these concepts:
{list(SERIES.keys())}

Return ONLY valid JSON with this exact structure:
{{
  "concepts": ["concept1", "concept2"],
  "task": "plot" or "regression",
  "explanation": "short plain-English sentence"
}}

Rules:
- Use only concepts from the approved list.
- If the question is about a relationship like Phillips Curve, use "regression".
- If the question is about comparing trends over time, use "plot".
- If the question mentions inflation, the student may later convert CPI to year-over-year inflation.
- Return only JSON.
"""

def get_plan(user_question, model="gpt-4.1-mini"):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": PLANNER_PROMPT},
            {"role": "user", "content": user_question}
        ],
        temperature=0
    )
    text = response.choices[0].message.content
    return json.loads(text)


## 6) Agent runner

This is the main function.

It:
1. gets a plan,
2. converts concepts to FRED series IDs,
3. builds the dataset,
4. either makes a plot or runs a simple regression.

You only need to make a few small edits in the TODO sections.


In [ ]:
def run_agent(question, start="1990-01-01"):
    plan = get_plan(question)
    print("PLAN:")
    print(json.dumps(plan, indent=2))

    concepts = plan["concepts"]
    task = plan["task"]

    # Convert concepts to series IDs
    series_ids = [SERIES[c] for c in concepts]
    df = build_dataset(series_ids, start=start)

    # Helpful transformations
    if "CPIAUCSL" in df.columns:
        df["inflation_yoy"] = yoy_pct(df["CPIAUCSL"])

    if "CPILFESL" in df.columns:
        df["core_inflation_yoy"] = yoy_pct(df["CPILFESL"])

    if "CES0500000003" in df.columns:
        df["wage_yoy"] = yoy_pct(df["CES0500000003"])

    # Simple decision rules
    if task == "plot":
        plot_cols = []

        for c in concepts:
            if c == "inflation" and "inflation_yoy" in df.columns:
                plot_cols.append("inflation_yoy")
            elif c == "core inflation" and "core_inflation_yoy" in df.columns:
                plot_cols.append("core_inflation_yoy")
            elif c == "wages" and "wage_yoy" in df.columns:
                plot_cols.append("wage_yoy")
            else:
                plot_cols.append(SERIES[c])

        make_plot(df, plot_cols, title=question)
        return {"plan": plan, "data": df.tail(), "note": plan["explanation"]}

    elif task == "regression":
        # Default simple Phillips Curve example
        if "inflation_yoy" not in df.columns or "UNRATE" not in df.columns:
            print("Not enough variables for the default regression.")
            return {"plan": plan, "data": df.tail(), "note": plan["explanation"]}

        reg_df = df[["inflation_yoy", "UNRATE"]].dropna().copy()
        result = run_regression(reg_df, "inflation_yoy ~ UNRATE")
        return {"plan": plan, "regression": result, "note": plan["explanation"]}

    else:
        print("Unknown task.")
        return {"plan": plan}


## 7) Required homework tasks

Run your agent on these 4 questions.

### Required prompts
1. `Is there evidence of a Phillips Curve since 1990?`  
2. `Compare oil prices and inflation after 2015.`  
3. `Did wages keep up with inflation after COVID?`  
4. `Compare federal funds rate and inflation since 2000.`

For each one, include:
- the plan,
- the output,
- a short written interpretation in **your own words**.


In [ ]:
# Prompt 1
result1 = run_agent("Is there evidence of a Phillips Curve since 1990?", start="1990-01-01")
result1


In [ ]:
# Prompt 2
result2 = run_agent("Compare oil prices and inflation after 2015.", start="2015-01-01")
result2


In [ ]:
# Prompt 3
result3 = run_agent("Did wages keep up with inflation after COVID?", start="2020-01-01")
result3


In [ ]:
# Prompt 4
result4 = run_agent("Compare federal funds rate and inflation since 2000.", start="2000-01-01")
result4


## 8) Short interpretation section

After each run, write 3–5 sentences in plain English:
- What variables did the agent choose?
- What did the plot or regression show?
- What is your economic interpretation?

This part matters.


### Student interpretation for Prompt 1
Write here.

### Student interpretation for Prompt 2
Write here.

### Student interpretation for Prompt 3
Write here.

### Student interpretation for Prompt 4
Write here.


## 9) Reflection questions

Answer these briefly:

1. Why do we use an approved list of series instead of letting the model search for anything?  
2. Why do we convert oil prices from daily to monthly before merging with monthly data?  
3. What is one strength and one weakness of your agent?


### Your answers
1.  
2.  
3.  


## 10) Suggested grading rubric

- Notebook runs correctly: 25  
- Agent gets and uses a valid plan: 20  
- Correct use of approved series and transformations: 20  
- Four required prompts completed: 20  
- Interpretations and reflection: 15
